# ablation

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.

The fixed TNG evaluation rows are included in TNG reference training: this comparison is not a held-out calibration test.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import time
import functools
from pathlib import Path
import torch
from scipy.stats import gaussian_kde
from scipy.special import rel_entr
from sklearn.preprocessing import StandardScaler
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
results_dir = configure_run(PROJECT)   
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"
results_dir.mkdir(parents=True, exist_ok=True)

def load_checkpoint_any_device(path, device):
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load
    return obj

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_posteriors = flam_ckpt["posteriors"]
flam_samples    = flam_ckpt["samples"]

tng_ckpt = load_checkpoint_any_device(results_dir / "tng_npe_checkpoint.pkl", device)
tng_posteriors = tng_ckpt["posteriors"]
tng_samples    = tng_ckpt["samples"]

with open(LEGACY_RESULTS_DIR / "optuna_search_results.pkl", "rb") as f:
    best_params = pickle.load(f)["best_params"]

with open(results_dir / "js_divergence_results.pkl", "rb") as f:
    js_results = pickle.load(f)

# raw data
with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)

keep_idx    = np.load(tng_dir / "keep_idx.npy")
tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_same_host   = np.load(tng_dir / "have_same_host.npy").astype(bool)

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
    sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
    kap_i, kap_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
    met_i = met_transform["flamingo_met_dex"][pi]
    met_j = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (sigma_i+sigma_j)/2, "sigma_diff": np.abs(sigma_i-sigma_j),
        "kappa_mean": (kap_i+kap_j)/2, "kappa_diff": np.abs(kap_i-kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
    })

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):
    gr_i, gr_j = tng_gr_sample[tng_pairs_i], tng_gr_sample[tng_pairs_j]
    met_raw_i, met_raw_j = tng_met_sample[tng_pairs_i], tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (tng_sigma_i+tng_sigma_j)/2, "sigma_diff": np.abs(tng_sigma_i-tng_sigma_j),
        "kappa_mean": (tng_kappa_i+tng_kappa_j)/2, "kappa_diff": np.abs(tng_kappa_i-tng_kappa_j),
        "separation": tng_separation, "vr": tng_vr, "vt": tng_vt,
        "log_mass_ratio": tng_log_mass_ratio, "log_m200c": tng_log_m200c,
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)

tng_df = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)
tng_df["same_host"] = tng_same_host

BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
}
TARGET = "log_m200c"

# we only use quantities which are calculated from position and velocities for inference.
# seperation 761 +- 11 kpc Li et al. 2021
# vr,vt van der Marel et al. 2012
# vt contested Salomon et al. 2021 give vt = 82.4 +-31.2 km/s. We run both
# log_mass_ratio: ratio of total bound masses not stellar. Sawala et al. 2023. M_mw = 1.1 +- 0.2
# M_31 = 1.5 +- 0.5 x 10^12 Msun.
# One thing to note though boundsubhalo/totalmass is not equal to observational m200 and does not cancel exactly in a ratio; this is a proxy assumption
LG_OBS, LG_UNC = lg_observation("vdM2012")

LG_SAFE_BLOCKS = ["core_dynamics"]

ALL_FEATURES = ["separation","vr","vt","log_mass_ratio","sigma_mean","sigma_diff",
                 "kappa_mean","kappa_diff","met_diff","gr_mean","gr_diff"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])

prior = BoxUniform(low=torch.tensor([11.5], device=device), high=torch.tensor([14.5], device=device))
xs = np.linspace(11.5, 14.5, 500)

def js_divergence(p_samples, q_samples, xs, base=2, bw_method=0.15):
    p = gaussian_kde(p_samples, bw_method=bw_method)(xs)
    q = gaussian_kde(q_samples, bw_method=bw_method)(xs)
    p, q = p/p.sum(), q/q.sum()
    m = 0.5*(p+q)
    return 0.5*np.sum(rel_entr(p,m))/np.log(base) + 0.5*np.sum(rel_entr(q,m))/np.log(base)

print(f"FLAMINGO: {len(flam_df_clean):,}   TNG: {len(tng_df_clean):,}")

In [ ]:
ABLATION_VARIANTS = {
    "core_chemistry":   {"feats": BLOCKS["core_chemistry"],  "covariate": False, "needs_reference": True},
    "core_photometry":  {"feats": BLOCKS["core_photometry"], "covariate": False, "needs_reference": True},
    "morphology_no_kappa":         {"feats": ["separation","vr","vt","log_mass_ratio","sigma_mean","sigma_diff"], "covariate": False, "needs_reference": True},
    "morphology_no_sigma":         {"feats": ["separation","vr","vt","log_mass_ratio","kappa_mean","kappa_diff"], "covariate": False, "needs_reference": True},
    "core_morphology":             {"feats": BLOCKS["core_morphology"], "covariate": False, "needs_reference": True},
    "morphology_same_host_covar":  {"feats": BLOCKS["core_morphology"], "covariate": True,  "needs_reference": True},
}


def build_arrays(df, feats, use_covariate, scaler=None, fit_scaler=False):
    X_raw = df[feats].values.astype(np.float32)
    if use_covariate:
        cov = df["same_host"].values.astype(np.float32).reshape(-1, 1)
        X_raw = np.hstack([X_raw, cov])
    if fit_scaler:
        scaler = StandardScaler()
        X_sc = scaler.fit_transform(X_raw)
    else:
        X_sc = scaler.transform(X_raw)
    y = df[TARGET].values.astype(np.float32)
    return X_sc, y, scaler

# We Fix the evaluation set. The TNG rows are held constant across all variants and both simulations. We only change the feature set.

N_EVAL   = 500
_eval_rng = np.random.default_rng(20260829)
EVAL_DF  = tng_df_clean.iloc[
    _eval_rng.choice(len(tng_df_clean), size=min(N_EVAL, len(tng_df_clean)), replace=False)
].reset_index(drop=True)

def posterior_draws_at(posterior, feats, use_covariate, scaler, df=EVAL_DF, n_draws=500):
    """(n_draws, n_rows) posterior samples, one column per evaluation point."""
    X_sc, _, _ = build_arrays(df, feats, use_covariate, scaler=scaler, fit_scaler=False)
    with torch.no_grad():
        s = posterior.sample_batched((n_draws,), x=torch.tensor(X_sc, device=device))
    return s.squeeze(-1).cpu().numpy()

def mean_w2(A, B):
    """Mean 1-D 2-Wasserstein distance between matched columns of A and B."""
    A = np.sort(A, axis=0); B = np.sort(B, axis=0)
    return float(np.mean(np.sqrt(np.mean((A - B) ** 2, axis=0))))

def train_one_model(df, feats, use_covariate, density_builder):
    X_sc, y, scaler = build_arrays(df, feats, use_covariate, fit_scaler=True)
    theta = torch.tensor(y.reshape(-1, 1), device=device)
    x     = torch.tensor(X_sc, device=device)
    inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
    inference.append_simulations(theta, x)
    density_est = inference.train(
        training_batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"],
        max_num_epochs=200, stop_after_epochs=20, show_train_summary=False,
    )
    posterior = inference.build_posterior(density_est)
    return posterior, scaler

In [ ]:
REFERENCE_CHECKPOINT = results_dir / "ablation_reference_checkpoint.pkl"

reference_posteriors = {}
reference_scalers    = {}
reference_samples    = {}   # draws at fixed TNG evaluation features, per variant/sim

if REFERENCE_CHECKPOINT.exists():
    with open(REFERENCE_CHECKPOINT, "rb") as f:
        ck = pickle.load(f)
    reference_posteriors.update(ck.get("posteriors", {}))
    reference_scalers.update(ck.get("scalers", {}))
    reference_samples.update(ck.get("samples", {}))
    print(f"Resuming: {len(reference_samples)} reference model(s) already done.")

density_builder = posterior_nn(
    model="maf",
    hidden_features=best_params["hidden_features"],
    num_transforms=best_params["num_transforms"],
)

legacy_reference = load_checkpoint_any_device(
    LEGACY_RESULTS_DIR / "ablation_reference_checkpoint.pkl", device)
for variant_name, spec in ABLATION_VARIANTS.items():
    if not spec["needs_reference"]:
        continue
    feats, use_cov = spec["feats"], spec["covariate"]

    for sim_name, df in [("flam", flam_df_clean), ("tng", tng_df_clean)]:
        needed_key = f"{variant_name}__{sim_name}"
        if needed_key in reference_samples:
            continue

        print(f"Training reference: {variant_name}__{sim_name}  (N={len(df):,})")
        t0 = time.time()
        if sim_name == "flam":
            posterior = legacy_reference["posteriors"][needed_key]
            scaler = legacy_reference["scalers"][needed_key]
        else:
            posterior, scaler = train_one_model(df, feats, use_cov, density_builder)

        key = f"{variant_name}__{sim_name}"
        reference_posteriors[key] = posterior
        reference_scalers[key]    = scaler
        reference_samples[key]    = posterior_draws_at(
            posterior, feats, use_cov, scaler
        )
        print(f"  {key}: mean posterior median = "
              f"{np.median(reference_samples[key]):.3f}")

        print(f"  took {(time.time()-t0)/60:.1f} min")
        with open(REFERENCE_CHECKPOINT, "wb") as f:
            pickle.dump({"posteriors": reference_posteriors, "scalers": reference_scalers,
                         "samples": reference_samples, "run_version": RUN_VERSION,
                         "evaluation_design": "fixed TNG features; TNG reference includes evaluation rows, not held-out validation"}, f)

print("\nReference models done.")

In [ ]:
BOOTSTRAP_CHECKPOINT = results_dir / "ablation_bootstrap_checkpoint.pkl"
N_REPEATS = 200

bootstrap_null = {}
if BOOTSTRAP_CHECKPOINT.exists():
    with open(BOOTSTRAP_CHECKPOINT, "rb") as f:
        bootstrap_null = pickle.load(f)
    if bootstrap_null:
        print(f"Resuming: {sum(len(v) for v in bootstrap_null.values())} repeat(s) already in checkpoint.")

with open(BOOTSTRAP_CHECKPOINT, "wb") as f:
    pickle.dump(bootstrap_null, f)

run_times = []

for variant_name, spec in ABLATION_VARIANTS.items():
    feats, use_cov = spec["feats"], spec["covariate"]

    flam_ref = reference_samples[f"{variant_name}__flam"]
    tng_ref  = reference_samples[f"{variant_name}__tng"]
    valid_flam = flam_df_clean[feats + [TARGET, "same_host"]].dropna()
    n_target = len(tng_df_clean[feats + [TARGET]].dropna())

    if variant_name not in bootstrap_null:
        bootstrap_null[variant_name] = []
    done = len(bootstrap_null[variant_name])

    density_builder = posterior_nn(model="maf", hidden_features=best_params["hidden_features"],
                                     num_transforms=best_params["num_transforms"])

    for rep in range(done, N_REPEATS):
        t0 = time.time()
        seed = 3000 + rep
        sample_df = valid_flam.sample(n=n_target, random_state=seed)
        torch.manual_seed(seed)
        np.random.seed(seed)

        posterior, scaler = train_one_model(sample_df, feats, use_cov, density_builder)
        sub = posterior_draws_at(posterior, feats, use_cov, scaler)

        entry = {"seed": seed,
                "w2_vs_flam": mean_w2(sub, flam_ref),
                "w2_vs_tng":  mean_w2(sub, tng_ref),
                "median":     float(np.median(sub))}

        bootstrap_null[variant_name].append(entry)

        elapsed = time.time() - t0
        run_times.append(elapsed)
        avg = sum(run_times) / len(run_times)
        remaining = sum(N_REPEATS - len(bootstrap_null.get(v, [])) for v in ABLATION_VARIANTS)
        print(f"{variant_name}  rep {rep+1}/{N_REPEATS}  ({elapsed/60:.1f} min, ETA {avg*remaining/60:.0f} min)")

        with open(BOOTSTRAP_CHECKPOINT, "wb") as f:
            pickle.dump(bootstrap_null, f)

print("\nAll variants done.")

In [ ]:
def summarize_arr(name, real_w2, null_w2s):
    finite_mask = np.isfinite(null_w2s)
    n_dropped = (~finite_mask).sum()
    null_w2s = null_w2s[finite_mask]

    p = (1 + np.count_nonzero(null_w2s >= real_w2)) / (len(null_w2s) + 1)
    ci = (np.percentile(null_w2s, 2.5), np.percentile(null_w2s, 97.5))
    dropped_note = f"  [{n_dropped} non-finite dropped]" if n_dropped else ""
    print(f"{name:<45} {real_w2:>18.4f} {np.median(null_w2s):>14.4f} "
          f"({ci[0]:.4f}, {ci[1]:.4f})   p={p:.4f}{dropped_note}")
    return {"real_w2": real_w2, "null_median": np.median(null_w2s), "ci95": ci,
            "p": p, "n_dropped": int(n_dropped), "n_used": len(null_w2s)}

print(f"{'Variant':<45} {'Real W2 (vs full)':>18} {'Null median':>14} {'Null 95% CI':>22} {'p-value':>10}")

final_summary = {}

for variant_name, spec in ABLATION_VARIANTS.items():
    reps = bootstrap_null[variant_name]
    if not reps:
        print(f"{variant_name:<45}  (no valid reps yet)")
        continue

    null_w2s = np.array([r["w2_vs_flam"] for r in reps])
    real_w2 = mean_w2(
        reference_samples[f"{variant_name}__flam"],
        reference_samples[f"{variant_name}__tng"]
    )
    final_summary[variant_name] = summarize_arr(
        variant_name, real_w2, null_w2s
    )

with open(results_dir / "ablation_bootstrap_summary.pkl", "wb") as f:
    pickle.dump(final_summary, f)